# Gipformer 1.5 68M + DeepFilterNet3 — Colab T4
Bản thử nghiệm riêng, dựa trên `Gipformer15_68M_T4(1).ipynb` và [inference PyTorch chính thức](https://github.com/ggroup-ai-lab/gipformer/blob/0cac5b183d2e1a9e3429ff69080e09cc62ebdf9b/infer_pytorch.py).
**T4 GPU → Run all**, không upload audio. Dùng checkpoint **gipformer1.5-68M-rnnt** và một model giữ trong bộ nhớ cho cả latency + quality.
Giữ dataset/seed/noise/SNR và cách tính WER/CER giống bản Zipformer: 461 đầu vào, 922 kết quả ASR ở cấu hình mặc định.
Kết quả Gipformer lưu riêng, không ghi đè báo cáo Zipformer.

Decoder mặc định **modified_beam_search, beam=4** như notebook tham chiếu. Zipformer đang dùng greedy search;
nếu so sánh cùng chế độ, đổi `DECODING_METHOD='greedy_search'`. Hai runtime PyTorch/ONNX vẫn khác nhau.
Gipformer dùng PyTorch CUDA trực tiếp, không kiểm tra GPU qua provider của Sherpa.
Bản này đã kiểm tra cấu trúc/cú pháp và logic CPU; chưa chạy end-to-end trên T4 tại môi trường biên soạn.

## 1. Cấu hình và kiểm tra GPU
Giữ `DATA_ROOT` mặc định để dùng lại dữ liệu/DF3 của notebook Zipformer nếu cùng runtime và manifest phù hợp.
Runtime mới sẽ tự tạo dataset. Có thể đổi đường dẫn để tạo bộ riêng; giữ cùng seed để so sánh.
`OUTPUT_ROOT` luôn tách khỏi dataset. `N_SPEECH=20` tương ứng 461 đầu vào; có thể chọn ít câu hơn trong thư mục dataset mới để thử nhanh.

In [ ]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
DATA_ROOT = Path('/content/zipformer30m_quality')
OUTPUT_ROOT = Path('/content/gipformer15_quality_results')
WORK_DIR = Path('/content/gipformer15_benchmark_workers')
GIP_REPO = Path('/content/gipformer-official')
GIP_ENV = Path('/content/gipformer-py312')
GIP_PY = GIP_ENV/'bin/python'
ICEFALL_DIR = Path('/content/gipformer-icefall')
GIP_COMMIT = '0cac5b183d2e1a9e3429ff69080e09cc62ebdf9b'
ICEFALL_COMMIT = '1c388ad9be94c0ee9de76f7a6ed45b1cff677d13'
MODEL_ID = 'g-group-ai-lab/gipformer1.5-68M-rnnt'
MODEL_REVISION = 'dd9227dcd8705c13f33bdbe59728d546ab94480f'
DECODING_METHOD, BEAM_SIZE = 'modified_beam_search', 4
N_SPEECH, DATA_SEED = 20, 20261019
PERF_SAMPLE_ID = 'vi_000_keyboard_events_snr05'
assert DATA_ROOT.resolve() != OUTPUT_ROOT.resolve()
for directory in (DATA_ROOT, OUTPUT_ROOT, WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## 2. Môi trường Gipformer PyTorch CUDA riêng
Dùng `uv sync --extra pytorch` theo repo chính thức, với Python 3.12.3 riêng.
Repo pin Torch/torchaudio 2.8.0 CUDA 12.8, k2 và kaldifeat tương ứng. Commit repo, Icefall và checkpoint được cố định.
Dependency Sherpa CPU trong repo không được dùng cho đường suy luận PyTorch này.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'uv', 'numpy', 'scipy', 'soundfile', 'huggingface-hub', 'ipywidgets'], check=True)
subprocess.run(['uv', 'python', 'install', '3.12.3'], check=True)

def checkout_pinned(url, directory, commit, sparse=None):
    if not (directory/'.git').exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(directory)], check=True)
    subprocess.run(['git', '-C', str(directory), 'fetch', '--depth', '1', 'origin', commit], check=True)
    if sparse:
        subprocess.run(['git', '-C', str(directory), 'sparse-checkout', 'set', *sparse], check=True)
    subprocess.run(['git', '-C', str(directory), 'checkout', '--detach', commit], check=True)

checkout_pinned('https://github.com/ggroup-ai-lab/gipformer.git', GIP_REPO, GIP_COMMIT)
checkout_pinned('https://github.com/k2-fsa/icefall.git', ICEFALL_DIR, ICEFALL_COMMIT,
                sparse=['icefall', 'egs/librispeech/ASR'])
uv_environment = os.environ.copy()
uv_environment['UV_PROJECT_ENVIRONMENT'] = str(GIP_ENV)
subprocess.run(['uv', 'sync', '--project', str(GIP_REPO), '--extra', 'pytorch',
                '--python', '3.12.3'], env=uv_environment, check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(GIP_PY), 'scipy'], check=True)
GIP_PROCESS_ENV = os.environ.copy()
GIP_PROCESS_ENV['CUDA_VISIBLE_DEVICES'] = '0'
GIP_PROCESS_ENV['OMP_NUM_THREADS'] = '2'
subprocess.run([str(GIP_PY), '-c',
    "import torch, k2, kaldifeat; "
    "assert torch.cuda.is_available(), 'Gipformer CUDA unavailable'; "
    "print('Torch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))"],
    env=GIP_PROCESS_ENV, check=True)

## 3. Tải checkpoint Gipformer **1.5**
Dùng `model.pt`, `bpe.model`, `tokens.txt` của [model chính thức](https://huggingface.co/g-group-ai-lab/gipformer1.5-68M-rnnt).
Không dùng ONNX hoặc checkpoint Gipformer v1. Worker kiểm tra model parameters, Fbank và encoder output thực sự nằm trên CUDA.

In [ ]:
from huggingface_hub import snapshot_download
model_dir = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION,
                    allow_patterns=['model.pt', 'bpe.model', 'tokens.txt']))
for name in ('model.pt', 'bpe.model', 'tokens.txt'):
    assert (model_dir/name).is_file(), name
print('Checkpoint:', model_dir)

## 4. Môi trường DeepFilterNet3 riêng
Giữ nguyên Python 3.12.3 + Torch/torchaudio 2.5.1 CUDA 12.1 và pyDF build bằng maturin của notebook Zipformer.
Các thư viện CUDA của DF3 và Gipformer được nạp ở hai worker riêng.

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

def run_cmd(args, env=None):
    args = [str(x) for x in args]
    print("\n+", " ".join(args), flush=True)

    subprocess.run(
        args,
        check=True,
        env=env
    )

run_cmd([
    sys.executable, "-m", "pip", "install", "-q",
    "uv",
    "maturin>=1.5,<2"
])

run_cmd([
    "uv", "python", "install", "3.12.3"
])

DF_ENV = Path("/content/df3-py312")
DF_PY = DF_ENV / "bin" / "python"

if not DF_PY.exists():
    run_cmd([
        "uv", "venv",
        "--seed",
        "--python", "3.12.3",
        str(DF_ENV)
    ])

run_cmd([
    DF_PY, "-c",
    "import sys; "
    "print('Python:', sys.version); "
    "assert sys.version_info[:3] == (3, 12, 3)"
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "torch==2.5.1",
    "torchaudio==2.5.1",

    "--index-url",
    "https://download.pytorch.org/whl/cu121"
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "numpy>=1.24,<2",
    "scipy",
    "soundfile>=0.10,<0.13",
    "loguru",
    "appdirs",
    "requests",
    "sympy",
    "packaging>=23,<25",
    "poetry-core>=1"
])

run_cmd([
    DF_PY, "-c",
    "import torch, torchaudio; "
    "print('PyTorch:', torch.__version__); "
    "print('Torchaudio:', torchaudio.__version__); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('CUDA version:', torch.version.cuda); "
    "assert torch.cuda.is_available(), 'CUDA is unavailable'"
])

rust_bin = Path.home() / ".cargo" / "bin"

build_env = os.environ.copy()

build_env["PATH"] = (
    str(rust_bin)
    + os.pathsep
    + build_env.get("PATH", "")
)

if shutil.which("cargo", path=build_env["PATH"]) is None:
    run_cmd([
        "bash", "-lc",
        'curl --proto "=https" --tlsv1.2 -sSf '
        'https://sh.rustup.rs | sh -s -- -y --profile minimal'
    ])

if (rust_bin / "rustup").exists():
    run_cmd([
        rust_bin / "rustup",
        "update",
        "stable"
    ], env=build_env)

run_cmd([
    "cargo", "--version"
], env=build_env)

repo = Path("/content/DeepFilterNet-src")

if not (repo / "pyDF" / "Cargo.toml").is_file():
    run_cmd([
        "git", "clone",
        "--depth", "1",
        "https://github.com/Rikorose/DeepFilterNet.git",
        str(repo)
    ])

wheels = Path("/content/df3-wheels-py312")
wheels.mkdir(parents=True, exist_ok=True)

libdf_wheels = sorted(
    wheels.glob("*deepfilterlib*cp312*.whl")
)

if not libdf_wheels:

    run_cmd([
        sys.executable,
        "-m", "maturin",
        "build",
        "--release",
        "--interpreter", str(DF_PY),
        "--manifest-path", str(repo / "pyDF" / "Cargo.toml"),
        "--out", str(wheels)
    ], env=build_env)

    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*cp312*.whl")
    )

if not libdf_wheels:
    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*.whl")
    )

assert libdf_wheels, (
    f"No deepfilterlib wheel found: "
    f"{list(wheels.glob('*.whl'))}"
)

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(libdf_wheels[-1])
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(repo / "DeepFilterNet")
])

verify_code = """
import sys
import torch
import torchaudio
import df
import libdf

from df.enhance import init_df, enhance

print("=" * 55)
print("DeepFilterNet3 Environment Verification")
print("=" * 55)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "DeepFilterNet3 requires CUDA"
)

print("GPU:", torch.cuda.get_device_name(0))
print("DeepFilterNet import: OK")
print("deepfilterlib import: OK")

print("=" * 55)
print("DeepFilterNet3 installation successful!")
print("=" * 55)
"""

run_cmd([
    DF_PY,
    "-c",
    verify_code
])

print("\nDeepFilterNet3 isolated Python 3.12.3 installation complete.")

## 5. Mã dataset và worker Gipformer
Notebook chứa sẵn mã worker, không cần upload `.py`.
Nguồn dữ liệu và cách trộn giữ nguyên: 6 loại nhiễu sự kiện + nhóm keyboard liên tục, SNR 0/5/10 dB,
20 câu sạch và 21 mẫu noise-only. SNR dùng frame năng lượng hoạt động, không dùng padding im lặng.

In [ ]:
QUALITY_WORKER_SOURCE = r'''
"""Vietnamese speech benchmark with continuous and intermittent office noise; also embedded in the Colab notebook."""
import argparse
import csv
import hashlib
import http.client
import io
import json
import math
import os
from pathlib import Path
import tarfile
import time
import unicodedata
import urllib.request
import urllib.error
import zipfile

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

FLEURS_REVISION = '70bb2e84b976b7e960aa89f1c648e09c59f894dd'
FLEURS_BASE = f'https://huggingface.co/datasets/google/fleurs/resolve/{FLEURS_REVISION}/data/vi_vn'
DCASE_URL = 'https://archive.org/download/dcase2016_task2_train_dev/dcase2016_task2_train_dev.zip'
DCASE_MD5 = '4e1b5e8887159193e8624dec801eb9e7'
# Storage hosts returned by archive.org/metadata/dcase2016_task2_train_dev.
DCASE_MIRRORS = [
    f'https://{host}/17/items/dcase2016_task2_train_dev/dcase2016_task2_train_dev.zip'
    for host in ('ia801900.us.archive.org', 'ia601900.us.archive.org')
]


NOISE_PREFIXES = {'keyboard': 'keyboard', 'knock': 'knock', 'doorslam': 'doorslam',
                  'keys': 'keysDrop', 'pageturn': 'pageturn', 'phone': 'phone'}
NOISE_CONDITIONS = [('keyboard', 'continuous')] + [(name, 'events') for name in NOISE_PREFIXES]


def normalize_text(text):
    text = unicodedata.normalize('NFC', text).casefold()
    return ' '.join(''.join(c if c.isalnum() or c.isspace() else ' ' for c in text).split())


def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, a in enumerate(reference, 1):
        current = [i]
        for j, b in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j-1] + (a != b)))
        previous = current
    return previous[-1]


def error_counts(reference, hypothesis):
    ref, hyp = normalize_text(reference), normalize_text(hypothesis)
    words, chars = ref.split(), ref.replace(' ', '')
    word_errors = edit_distance(words, hyp.split())
    char_errors = edit_distance(chars, hyp.replace(' ', ''))
    return dict(reference_normalized=ref, hypothesis_normalized=hyp,
                word_errors=word_errors, reference_words=len(words),
                char_errors=char_errors, reference_chars=len(chars),
                wer=word_errors/len(words) if words else None,
                cer=char_errors/len(chars) if chars else None, has_text=bool(hyp))


def mix_at_snr(speech, noise, snr_db, *, active_mask=None, speech_mask=None):
    speech, noise = np.asarray(speech, dtype=np.float64), np.asarray(noise, dtype=np.float64)
    if speech.ndim != 1 or speech.shape != noise.shape or not speech.size:
        raise ValueError('Speech/noise must be nonempty mono arrays of identical length')
    if not np.isfinite(speech).all() or not np.isfinite(noise).all() or not math.isfinite(snr_db):
        raise ValueError('Nonfinite audio or SNR')
    active_mask = np.ones(len(noise), dtype=bool) if active_mask is None else np.asarray(active_mask, dtype=bool)
    speech_mask = np.ones(len(speech), dtype=bool) if speech_mask is None else np.asarray(speech_mask, dtype=bool)
    if (active_mask.shape != speech.shape or speech_mask.shape != speech.shape
            or not active_mask.any() or not speech_mask.any()):
        raise ValueError('Invalid or empty activity mask')
    speech_rms = np.sqrt(np.mean(speech[speech_mask]**2))
    noise_rms = np.sqrt(np.mean(noise[active_mask]**2))
    if min(speech_rms, noise_rms) < 1e-10:
        raise ValueError('Cannot mix silent speech/noise at a defined SNR')
    scaled_noise = noise * (speech_rms / noise_rms) * 10**(-snr_db/20)
    mixed = speech + scaled_noise
    gain = min(1., .95/max(float(np.max(np.abs(mixed))), 1e-10))
    target = speech * gain
    mixed = mixed * gain
    actual_snr = 10*np.log10(np.mean(target[speech_mask]**2)/np.mean((scaled_noise[active_mask]*gain)**2))
    return mixed.astype(np.float32), target.astype(np.float32), float(actual_snr), gain


def summarize(rows):
    groups = {}
    for row in rows:
        groups.setdefault((row['kind'], row.get('noise_type'), row.get('noise_style'),
                           row['snr_db'], row['branch']), []).append(row)
    result = []
    for (kind, noise_type, noise_style, snr_db, branch), items in groups.items():
        words, chars = sum(x['reference_words'] for x in items), sum(x['reference_chars'] for x in items)
        seconds = sum(x['audio_seconds'] for x in items)
        group = dict(kind=kind, noise_type=noise_type, noise_style=noise_style,
                     snr_db=snr_db, branch=branch, clips=len(items),
                     reference_words=words, reference_chars=chars,
                     wer=sum(x['word_errors'] for x in items)/words if words else None,
                     cer=sum(x['char_errors'] for x in items)/chars if chars else None,
                     nonempty_transcript_rate=sum(x['has_text'] for x in items)/len(items),
                     median_asr_seconds=float(np.median([x['asr_seconds'] for x in items])),
                     aggregate_asr_rtf=sum(x['asr_seconds'] for x in items)/seconds)
        if branch == 'df3':
            estimates = [x['asr_seconds'] + x['enhancement_seconds'] + x['resample_seconds'] for x in items]
            group['median_pipeline_estimate_seconds'] = float(np.median(estimates))
        result.append(group)
    return result


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_audio(source, target_sr=48000):
    wave, sr = sf.read(source, dtype='float32', always_2d=True)
    wave = wave.mean(axis=1)
    if wave.size == 0 or not np.isfinite(wave).all():
        raise ValueError('Empty or nonfinite audio')
    if sr != target_sr:
        divisor = math.gcd(sr, target_sr)
        wave = resample_poly(wave, target_sr//divisor, sr//divisor).astype(np.float32)
    return wave, sr


def download(url, path, *, fallback_urls=(), expected_md5=None, attempts=3):
    path = Path(path)
    if attempts < 1:
        raise ValueError('attempts must be positive')
    def checksum_ok(candidate):
        if expected_md5 is None:
            return True
        digest = hashlib.md5()
        with candidate.open('rb') as f:
            while chunk := f.read(1024*1024):
                digest.update(chunk)
        return digest.hexdigest() == expected_md5
    if path.exists() and checksum_ok(path):
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    urls = list(dict.fromkeys([url, *fallback_urls]))
    last_error = None
    for attempt in range(attempts):
        for source_url in urls:
            print(f'Downloading ({attempt+1}/{attempts}): {source_url}', flush=True)
            try:
                request = urllib.request.Request(source_url, headers={'User-Agent': 'ZipformerBenchmark/1.0'})
                with urllib.request.urlopen(request, timeout=60) as source, temporary.open('wb') as destination:
                    expected_size = source.headers.get('Content-Length')
                    received = 0
                    while chunk := source.read(1024*1024):
                        destination.write(chunk)
                        received += len(chunk)
                if expected_size is not None and received != int(expected_size):
                    raise OSError(f'Incomplete download: {received}/{expected_size} bytes')
                if not checksum_ok(temporary):
                    raise OSError('Downloaded file checksum mismatch')
                temporary.replace(path)
                return path
            except (urllib.error.URLError, OSError, http.client.HTTPException) as error:
                last_error = error
                temporary.unlink(missing_ok=True)
                print(f'Download failed: {error}', flush=True)
        if attempt + 1 < attempts:
            time.sleep(2**attempt)
    raise RuntimeError(f'Cannot download {url} after {attempts} attempts across {len(urls)} URLs. '
                       f'Last error: {last_error}. You may place the original file at {path}; '
                       'it will be reused only if its checksum is valid.') from last_error


def noise_segment(wave, length, rng):
    offset = int(rng.integers(len(wave)))
    # Circular crop; reuses short isolated keyboard recordings without introducing zero pads.
    segment = wave[(offset + np.arange(length)) % len(wave)]
    if np.sqrt(np.mean(segment.astype(float)**2)) < 1e-9:
        raise ValueError('Selected noise segment is silent')
    return segment, offset


def frame_activity(wave, sr, threshold_db=-35):
    """20ms RMS gate, an energy proxy rather than a speech VAD."""
    wave = np.asarray(wave, dtype=float)
    if wave.ndim != 1 or not wave.size or not np.isfinite(wave).all() or sr < 1:
        raise ValueError('Invalid waveform/sample rate')
    hop = max(1, round(.02*sr))
    padded = np.pad(wave, (0, (-len(wave)) % hop))
    rms = np.sqrt(np.mean(padded.reshape(-1, hop)**2, axis=1))
    active = rms > max(float(rms.max())*10**(threshold_db/20), 1e-10)
    return np.repeat(active, hop)[:len(wave)]


def activity_intervals(mask, sr):
    edges = np.diff(np.r_[False, mask, False].astype(np.int8))
    return [[int(start)/sr, int(stop)/sr] for start, stop in
            zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1))]


def build_noise_track(bank, length, sr, rng, noise_type, style, speech_mask):
    if not bank or style not in ('events', 'continuous') or length < 1:
        raise ValueError('Invalid noise bank, style, or track length')
    speech_mask = np.asarray(speech_mask, dtype=bool)
    if speech_mask.shape != (length,):
        raise ValueError('Speech activity length mismatch')
    if style == 'continuous':
        source = bank[int(rng.integers(len(bank)))]
        track, offset = noise_segment(source['wave'], length, rng)
        return track, frame_activity(track, sr, -45), [dict(
            start_sample=0, end_sample=length, source_file=source['file'], source_offset=offset,
            onset_region='unconstrained', requested_onset='unconstrained',
            speech_overlap_fraction=float(speech_mask.mean()))]
    track = np.zeros(length, dtype=np.float32)
    events = []
    gap, margin, hop = round(.3*sr), round(.1*sr), max(1, round(.02*sr))
    count = int(rng.integers(2, 4)) if noise_type == 'keyboard' else int(rng.integers(1, 4))
    if noise_type == 'phone':
        count = 1
    first_region = int(rng.integers(2))
    for index in range(count):
        source = bank[int(rng.integers(len(bank)))]
        offset = 0
        if noise_type == 'keyboard':
            size = min(round(float(rng.uniform(.5, 2))*sr), length-2*margin-gap)
            if size < 1:
                break
            event_wave, offset = noise_segment(source['wave'], size, rng)
            # Fade only synthetic keyboard burst boundaries, not transient source events.
            event_wave = event_wave.copy()
            fade = min(round(.005*sr), size//2)
            if fade:
                event_wave[:fade] *= np.linspace(0, 1, fade)
                event_wave[-fade:] *= np.linspace(1, 0, fade)
        elif noise_type == 'phone':
            # Preserve recorded rings; separate repeated rings by a natural quiet gap.
            pieces = []
            for ring in range(int(rng.integers(2, 4))):
                if ring:
                    pieces.append(np.zeros(round(.6*sr), dtype=np.float32))
                pieces.append(source['wave'])
            event_wave = np.concatenate(pieces)
            if len(event_wave) > min(5*sr, length-2*margin-gap):
                event_wave = source['wave']  # Use one complete ring instead of cutting its tail.
        else:
            event_wave = source['wave']  # Keep the full impact/page-turn and its decay.
        size = len(event_wave)
        if size > length-2*margin-gap:
            continue
        candidates = np.arange(margin, length-margin-size+1, hop)
        for previous in events:
            candidates = candidates[(candidates+size+gap <= previous['start_sample']) |
                                    (candidates >= previous['end_sample']+gap)]
        if not candidates.size:
            continue
        requested = 'speech' if (index+first_region) % 2 == 0 else 'pause'
        preferred = candidates[speech_mask[candidates] == (requested == 'speech')]
        onset_region = requested if preferred.size else 'unconstrained'
        start = int(rng.choice(preferred if preferred.size else candidates))
        stop = start+size
        track[start:stop] = event_wave
        event_mask = frame_activity(event_wave, sr, -45)
        events.append(dict(start_sample=start, end_sample=stop, source_file=source['file'],
                           source_offset=offset, requested_onset=requested, onset_region=onset_region,
                           speech_overlap_fraction=float(speech_mask[start:stop][event_mask].mean())))
    if not events:
        raise ValueError(f'No complete {noise_type} event fits this utterance')
    return track, frame_activity(track, sr, -45), sorted(events, key=lambda e: e['start_sample'])


def prepare(root, count=20, seed=20261019):
    if count < 1:
        raise ValueError('count must be positive')
    root = Path(root)
    cache = root/'cache'
    cache.mkdir(parents=True, exist_ok=True)
    for folder in ('input_48k', 'raw_16k', 'df3_16k', 'noise_sources'):
        (root/folder).mkdir(exist_ok=True)
    rng = np.random.default_rng(seed)
    archive = download(DCASE_URL, cache/'dcase2016_task2_train_dev.zip',
                       fallback_urls=DCASE_MIRRORS, expected_md5=DCASE_MD5)
    banks = {name: [] for name in NOISE_PREFIXES}
    with zipfile.ZipFile(archive) as z:
        members = sorted(n for n in z.namelist() if n.lower().endswith('.wav') and 'train/' in n)
        for noise_type, prefix in NOISE_PREFIXES.items():
            selected = [n for n in members if Path(n).stem.lower().startswith(prefix.lower())]
            for index, member in enumerate(selected):
                payload = z.read(member)
                wave, native_sr = read_audio(io.BytesIO(payload))
                if np.sqrt(np.mean(wave.astype(float)**2)) < 1e-9:
                    continue
                name = f'{noise_type}_{index:02d}.wav'
                sf.write(root/'noise_sources'/name, wave, 48000, subtype='FLOAT')
                banks[noise_type].append(dict(wave=wave, member=member, native_sr=native_sr,
                      sha256=hashlib.sha256(payload).hexdigest(), file=name, noise_type=noise_type))
            if not banks[noise_type]:
                raise RuntimeError(f'No valid {noise_type} recordings in DCASE archive')
    tsv = download(FLEURS_BASE+'/test.tsv', cache/'fleurs_vi_test.tsv')
    references = {}
    with tsv.open(encoding='utf-8') as f:
        for fields in csv.reader(f, delimiter='\t', quoting=csv.QUOTE_NONE):
            if len(fields) < 7:
                raise ValueError('Unexpected FLEURS TSV schema')
            references[fields[1]] = dict(prompt_id=fields[0], text=fields[2], gender=fields[6])
    speech = []
    used_prompts = set()
    # Stream only the required prefix of the 544 MB archive instead of downloading all speech.
    print('Streaming FLEURS Vietnamese test audio...', flush=True)
    with urllib.request.urlopen(FLEURS_BASE+'/audio/test.tar.gz', timeout=60) as response:
        with tarfile.open(fileobj=response, mode='r|gz') as tar:
            for member in tar:
                name = Path(member.name).name
                meta = references.get(name)
                if not member.isfile() or meta is None or meta['prompt_id'] in used_prompts:
                    continue
                extracted = tar.extractfile(member)
                if extracted is None:
                    continue
                payload = extracted.read()
                wave, native_sr = read_audio(io.BytesIO(payload))
                if not 3 <= len(wave)/48000 <= 15:
                    continue  # Never truncate speech: transcript stays aligned with the full recording.
                wave = wave * (.05/max(float(np.sqrt(np.mean(wave.astype(float)**2))), 1e-10))
                wave *= min(1., .95/max(float(np.max(np.abs(wave))), 1e-10))
                speech.append(dict(meta, wave=wave.astype(np.float32), native_sr=native_sr,
                                   source_file=name, sha256=hashlib.sha256(payload).hexdigest()))
                used_prompts.add(meta['prompt_id'])
                print(f'Selected {len(speech)}/{count}: {name} ({len(wave)/48000:.1f}s, native {native_sr}Hz)', flush=True)
                if len(speech) == count:
                    break
    if len(speech) != count:
        raise RuntimeError(f'Only found {len(speech)} valid unique FLEURS prompts')
    rows = []
    def write_input(identifier, wave, **metadata):
        input_file = f'input_48k/{identifier}.wav'
        raw_file = f'raw_16k/{identifier}.wav'
        sf.write(root/input_file, wave, 48000, subtype='FLOAT')
        sf.write(root/raw_file, resample_poly(wave, 1, 3), 16000, subtype='FLOAT')
        rows.append(dict(id=identifier, input_48k=input_file, raw_16k=raw_file,
                         df3_16k=f'df3_16k/{identifier}.wav', audio_seconds=len(wave)/48000, **metadata))
    for index, sample in enumerate(speech):
        identifier = f'vi_{index:03d}'
        speech_mask = frame_activity(sample['wave'], 48000)
        common = {k: v for k, v in sample.items() if k != 'wave'}
        common['speech_active_intervals'] = activity_intervals(speech_mask, 48000)
        write_input(identifier+'_clean', sample['wave'], kind='clean', snr_db=None,
                    noise_type=None, noise_style='clean', events=[], noise_active_intervals=[],
                    achieved_snr_db=None, mixture_gain=1., **common)
        for noise_type, style in NOISE_CONDITIONS:
            track, mask, events = build_noise_track(banks[noise_type], len(sample['wave']), 48000,
                                                    rng, noise_type, style, speech_mask)
            for snr in (0, 5, 10):
                mixed, _, actual_snr, gain = mix_at_snr(sample['wave'], track, snr,
                                                       active_mask=mask, speech_mask=speech_mask)
                write_input(identifier+f'_{noise_type}_{style}_snr{snr:02d}', mixed,
                            kind='noisy', snr_db=snr, noise_type=noise_type, noise_style=style,
                            events=events, noise_active_intervals=activity_intervals(mask, 48000),
                            achieved_snr_db=actual_snr, mixture_gain=gain, **common)
    for noise_type, style in NOISE_CONDITIONS:
        for index in range(3):
            wave, mask, events = build_noise_track(banks[noise_type], 10*48000, 48000, rng,
                                      noise_type, style, np.zeros(10*48000, dtype=bool))
            wave = wave * (.05/float(np.sqrt(np.mean(wave[mask].astype(float)**2))))
            wave *= min(1., .95/max(float(np.max(np.abs(wave))), 1e-10))
            write_input(f'{noise_type}_{style}_only_{index:02d}', wave, kind='noise_only',
                        snr_db=None, text='', prompt_id=None, noise_type=noise_type, noise_style=style,
                        events=events, noise_active_intervals=activity_intervals(mask, 48000),
                        speech_active_intervals=[], achieved_snr_db=None, mixture_gain=None)
    manifest = dict(schema_version=2, seed=seed, speech_clips=count, snrs_db=[0, 5, 10], samples=rows,
                    noise_conditions=NOISE_CONDITIONS,
                    fleurs_revision=FLEURS_REVISION, fleurs_source=FLEURS_BASE,
                    noise_source=DCASE_URL, noise_md5=DCASE_MD5,
                    noise_recordings=[{k:v for k,v in item.items() if k!='wave'}
                                      for bank in banks.values() for item in bank],
                    licenses={'FLEURS': 'CC BY 4.0', 'DCASE2016 archive': 'CC BY 3.0'},
                    snr_definition='RMS of active speech across utterance / RMS of active noise events; '
                        '20ms energy gates: speech -35dB, noise -45dB relative to each track peak. '
                        'Not a true VAD. Same event positions across SNRs, joint anti-clipping gain.',
                    notes='First unique full 3-15s prompts in archive order. Event onset labels use '
                          'energy activity; events may extend into speech from a pause onset. '
                          'Inspect speech_overlap_fraction. Continuous keyboard is a separate condition.')
    save_json(root/'manifest.json', manifest)
    print('Prepared', len(rows), 'inputs:', count, 'clean,',
          count*len(NOISE_CONDITIONS)*3, 'noisy,', 3*len(NOISE_CONDITIONS), 'noise-only', flush=True)


def enhance_dataset(root):
    import torch
    from df.enhance import init_df, enhance
    from importlib.metadata import version
    if not torch.cuda.is_available():
        raise RuntimeError('DF3 requires CUDA')
    root = Path(root)
    manifest = read_json(root/'manifest.json')
    model, state, _, _ = init_df(model_base_dir='DeepFilterNet3', log_file=None)
    if next(model.parameters()).device.type != 'cuda' or state.sr() != 48000:
        raise RuntimeError('DF3 model must be CUDA / 48kHz')
    model.eval()
    first, _ = read_audio(root/manifest['samples'][0]['input_48k'])
    with torch.inference_mode():
        for _ in range(2):
            enhance(model, state, torch.from_numpy(first).unsqueeze(0), pad=True)
        torch.cuda.synchronize()
        metrics = {}
        for index, sample in enumerate(manifest['samples']):
            wave, _ = read_audio(root/sample['input_48k'])
            tensor = torch.from_numpy(np.ascontiguousarray(wave)).unsqueeze(0)
            torch.cuda.synchronize()
            start = time.perf_counter()
            cleaned = enhance(model, state, tensor, pad=True)
            torch.cuda.synchronize()
            seconds = time.perf_counter()-start
            array = cleaned.squeeze(0).detach().cpu().numpy()
            if array.shape != wave.shape or not np.isfinite(array).all():
                raise RuntimeError('DF3 returned invalid waveform/length')
            start = time.perf_counter()
            output = resample_poly(array, 1, 3).astype(np.float32)
            resample_seconds = time.perf_counter()-start
            sf.write(root/sample['df3_16k'], output, 16000, subtype='FLOAT')
            input_rms = float(np.sqrt(np.mean(wave.astype(float)**2)))
            output_rms = float(np.sqrt(np.mean(array.astype(float)**2)))
            metrics[sample['id']] = dict(enhancement_seconds=seconds, resample_seconds=resample_seconds,
                output_peak=float(np.max(np.abs(output))),
                rms_attenuation_db=20*math.log10(max(input_rms, 1e-12)/max(output_rms, 1e-12)))
            print(f'DF3 {index+1}/{len(manifest["samples"])}: {sample["id"]}', flush=True)
    save_json(root/'enhancement.json', dict(device='cuda', gpu=torch.cuda.get_device_name(0),
              torch_version=torch.__version__, deepfilternet_version=version('deepfilternet'),
              manifest_sha256=hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest(),
              samples=metrics, notes='Two warmups; one timed enhancement/clip, CPU frontend/synthesis included; resampling separate.'))


def recognize_dataset(root, model_dir, tokens, profile_dir):
    import ctypes
    import sherpa_onnx
    from importlib.metadata import version
    # Reuse the exact tested CUDA profile guard from the original worker.
    from asr_worker import inspect_cuda_profiles
    evidence = inspect_cuda_profiles(sorted(Path(profile_dir).glob('sherpa*.json')))
    root, model_dir = Path(root), Path(model_dir)
    manifest = read_json(root/'manifest.json')
    enhancement = read_json(root/'enhancement.json')
    expected = {sample['id'] for sample in manifest['samples']}
    if (set(enhancement['samples']) != expected or enhancement.get('manifest_sha256') !=
            hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest()):
        raise RuntimeError('Enhancement/manifest IDs differ; rerun DF3 for this dataset')
    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
        tokens=str(tokens), encoder=str(model_dir/'encoder-epoch-20-avg-10.onnx'),
        decoder=str(model_dir/'decoder-epoch-20-avg-10.onnx'),
        joiner=str(model_dir/'joiner-epoch-20-avg-10.onnx'), num_threads=2,
        sample_rate=16000, feature_dim=80, decoding_method='greedy_search', provider='cuda')
    cudart = ctypes.CDLL('libcudart.so.12')
    cudart.cudaDeviceSynchronize.restype = ctypes.c_int
    def recognize(wave):
        if cudart.cudaDeviceSynchronize():
            raise RuntimeError('CUDA synchronization failed')
        start = time.perf_counter()
        stream = recognizer.create_stream()
        stream.accept_waveform(16000, wave)
        recognizer.decode_stream(stream)
        text = stream.result.text
        if cudart.cudaDeviceSynchronize():
            raise RuntimeError('CUDA synchronization failed')
        return text, time.perf_counter()-start
    for _ in range(5):
        for name in ('raw_16k', 'df3_16k'):
            wave, _ = read_audio(root/manifest['samples'][0][name], 16000)
            recognize(wave)
    rows = []
    for index, sample in enumerate(manifest['samples']):
        pair = {}
        for branch in ('raw', 'df3'):
            wave, native_sr = read_audio(root/sample[branch+'_16k'], 16000)
            if native_sr != 16000:
                raise RuntimeError('ASR input must already be 16 kHz')
            pair[branch] = wave
        if len(pair['raw']) != len(pair['df3']):
            raise RuntimeError('Raw/DF3 lengths differ')
        for branch in (('raw', 'df3') if index % 2 == 0 else ('df3', 'raw')):
            text, seconds = recognize(pair[branch])
            row = dict(sample, branch=branch, transcript=text, asr_seconds=seconds,
                       **error_counts(sample['text'], text))
            if branch == 'df3':
                row.update(enhancement['samples'][sample['id']])
            rows.append(row)
        print(f'ASR {index+1}/{len(manifest["samples"])}: {sample["id"]}', flush=True)
    summary = summarize(rows)
    save_json(root/'quality_results.json', dict(samples=rows, summary=summary,
        cuda_probe_executed_node_counts=evidence, sherpa_version=version('sherpa-onnx'),
        model_revision=model_dir.name, df3_gpu=enhancement['gpu'],
        notes='Corpus WER/CER sum edit counts before division; casefold+NFC, punctuation removed, tones retained, digits not verbalized. Vietnamese WER uses whitespace tokens; CER excludes spaces. Empty-reference noise clips excluded from WER/CER, reported by nonempty transcript rate. One ASR run/clip; latency aggregates different clips, not repeated performance runs. Pipeline latency is a sum across warm separate workers, not measured end-to-end.'))
    for name, data in [('quality_per_clip.csv', rows), ('quality_summary.csv', summary)]:
        keys = list(dict.fromkeys(k for row in data for k in row))
        with (root/name).open('w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('mode', choices=['prepare', 'df3', 'asr'])
    p.add_argument('--root', required=True)
    p.add_argument('--count', type=int, default=20)
    p.add_argument('--seed', type=int, default=20261019)
    p.add_argument('--model-dir')
    p.add_argument('--tokens')
    p.add_argument('--profile-dir')
    args = p.parse_args()
    if args.mode == 'prepare':
        prepare(args.root, args.count, args.seed)
    elif args.mode == 'df3':
        enhance_dataset(args.root)
    else:
        if not all((args.model_dir, args.tokens, args.profile_dir)):
            p.error('asr requires --model-dir, --tokens, --profile-dir')
        recognize_dataset(args.root, args.model_dir, args.tokens, args.profile_dir)


if __name__ == '__main__':
    main()
'''

In [ ]:
GIP_WORKER_SOURCE = r'''
"""Gipformer 1.5 CUDA benchmark, using the official PyTorch inference recipe."""
import argparse
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import subprocess
import sys
import time

import numpy as np
from quality_benchmark import read_json, save_json, read_audio, error_counts, summarize

MODEL_ID = 'g-group-ai-lab/gipformer1.5-68M-rnnt'


def write_quality_reports(output, rows, metadata):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    summary = summarize(rows)
    save_json(output/'quality_results.json', dict(metadata, samples=rows, summary=summary,
        notes='Shared normalization and corpus WER/CER with Zipformer. NFC/casefold, punctuation removed, '
              'Vietnamese tones retained; digits not verbalized. WER uses whitespace tokens; CER excludes spaces. '
              'Noise-only clips have no WER/CER. Pipeline sums use separate warm DF3 and ASR workers, '
              'not measured end-to-end. Quality ASR timing is one run per variable-length clip.'))
    for filename, data in [('quality_per_clip.csv', rows), ('quality_summary.csv', summary)]:
        keys = list(dict.fromkeys(k for row in data for k in row))
        with (output/filename).open('w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)


class Gipformer:
    def __init__(self, repo, icefall_dir, model_dir, decoding_method, beam_size):
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError('Gipformer requires CUDA; CPU fallback is not allowed')
        self.torch = torch
        self.device = torch.device('cuda:0')
        sys.path.insert(0, str(repo))
        from infer_pytorch import setup_icefall, load_model_files
        setup_icefall(Path(icefall_dir))
        import k2
        import kaldifeat
        import sentencepiece as spm
        from beam_search import greedy_search_batch, modified_beam_search
        from torch.nn.utils.rnn import pad_sequence
        from train import add_model_arguments, get_model, get_params
        # Icefall sets inter-op once at import. Only intra-op can be adjusted here.
        torch.set_num_threads(2)
        self.greedy = greedy_search_batch
        self.beam_search = modified_beam_search
        self.pad_sequence = pad_sequence
        parser = argparse.ArgumentParser()
        parser.add_argument('--context-size', type=int, default=2)
        add_model_arguments(parser)
        old_git_dir = os.environ.get('GIT_DIR')
        try:
            os.environ['GIT_DIR'] = str(Path(icefall_dir)/'.git')
            params = get_params()
        finally:
            if old_git_dir is None:
                os.environ.pop('GIT_DIR', None)
            else:
                os.environ['GIT_DIR'] = old_git_dir
        params.update(vars(parser.parse_args([])))
        params.decoding_method, params.beam_size = decoding_method, beam_size
        paths = load_model_files(version='1.5', model_dir=str(model_dir))
        tokens = k2.SymbolTable.from_file(paths['tokens'])
        params.blank_id, params.unk_id = tokens['<blk>'], tokens['<unk>']
        count = sum(not symbol.startswith('#') for symbol in tokens.symbols)
        if params.blank_id == 0:
            count -= 1
        params.vocab_size = count+1  # Same vocabulary calculation as the official script.
        load_start = time.perf_counter()
        self.model = get_model(params)
        checkpoint = torch.load(paths['checkpoint'], map_location='cpu', weights_only=False)
        state = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
        incompatible = self.model.load_state_dict(state, strict=False)
        if incompatible.missing_keys:
            raise RuntimeError(f'Checkpoint is missing model parameters: {incompatible.missing_keys[:20]}')
        self.model.to(self.device).eval()
        if not all(p.device.type == 'cuda' for p in self.model.parameters()):
            raise RuntimeError('Some Gipformer parameters are not on CUDA')
        torch.cuda.synchronize()
        self.model_load_seconds = time.perf_counter()-load_start
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(paths['bpe_model'])
        opts = kaldifeat.FbankOptions()
        opts.device = self.device
        opts.frame_opts.dither = 0
        opts.frame_opts.snip_edges = False
        opts.frame_opts.samp_freq = 16000
        opts.mel_opts.num_bins = 80
        opts.mel_opts.high_freq = -400
        self.fbank = kaldifeat.Fbank(opts)
        self.decoding_method, self.beam_size = decoding_method, beam_size
        self.metadata = dict(model=MODEL_ID, model_version='1.5', runtime='official PyTorch CUDA recipe',
            model_revision=Path(model_dir).name, torch_version=torch.__version__, torch_cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0), k2_version=getattr(k2, '__version__', getattr(k2, '__dev_version__', None)),
            decoding_method=decoding_method, beam_size=beam_size if decoding_method=='modified_beam_search' else None,
            context_size=2, num_threads=torch.get_num_threads(),
            num_interop_threads=torch.get_num_interop_threads(), model_parameters=sum(p.numel() for p in self.model.parameters()),
            model_load_seconds=self.model_load_seconds,
            checkpoint_unexpected_keys=list(incompatible.unexpected_keys),
            cuda_checks=dict(model_parameters=True, features=False, encoder_output=False),
            repo_commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip(),
            icefall_commit=subprocess.check_output(['git','-C',str(icefall_dir),'rev-parse','HEAD'], text=True).strip())
        print(json.dumps(self.metadata, ensure_ascii=False, indent=2), flush=True)

    def recognize(self, wave):
        torch = self.torch
        if wave.ndim != 1 or not wave.size or not np.isfinite(wave).all():
            raise ValueError('Invalid waveform')
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.inference_mode():
            tensor = torch.from_numpy(np.ascontiguousarray(wave, dtype=np.float32)).to(self.device)
            features = self.fbank([tensor])
            lengths = torch.tensor([features[0].size(0)], device=self.device)
            features = self.pad_sequence(features, batch_first=True, padding_value=math.log(1e-10))
            if features.device.type != 'cuda':
                raise RuntimeError('Fbank features are not on CUDA')
            encoded, encoded_lengths = self.model.forward_encoder(features, lengths)
            if encoded.device.type != 'cuda':
                raise RuntimeError('Encoder output is not on CUDA')
            if self.decoding_method == 'greedy_search':
                hypotheses = self.greedy(model=self.model, encoder_out=encoded,
                                        encoder_out_lens=encoded_lengths)
            else:
                hypotheses = self.beam_search(model=self.model, encoder_out=encoded,
                                  encoder_out_lens=encoded_lengths, beam=self.beam_size)
            text = self.sp.decode(hypotheses[0])
        torch.cuda.synchronize()
        elapsed = time.perf_counter()-start
        self.metadata['cuda_checks'].update(features=True, encoder_output=True)
        return text, elapsed


def load_pair(root, sample):
    pair = {}
    for branch in ('raw', 'df3'):
        wave, sr = read_audio(Path(root)/sample[branch+'_16k'], 16000)
        if sr != 16000:
            raise RuntimeError('ASR files must already be 16kHz')
        pair[branch] = wave
    if len(pair['raw']) != len(pair['df3']):
        raise RuntimeError('Raw/DF3 lengths differ')
    return pair


def warmup(backend, pair):
    for _ in range(5):
        for wave in pair.values():
            backend.recognize(wave)


def performance(backend, root, output, sample, enhancement):
    pair = load_pair(root, sample)
    warmup(backend, pair)
    torch = backend.torch
    torch.cuda.reset_peak_memory_stats()
    results = {'raw': [], 'df3': []}
    for index in range(20):
        for branch in (('raw','df3') if index % 2 == 0 else ('df3','raw')):
            results[branch].append(backend.recognize(pair[branch]))
    for branch, runs in results.items():
        latencies = [elapsed for text, elapsed in runs]
        seconds = len(pair[branch])/16000
        median = float(np.median(latencies))
        p95 = float(np.percentile(latencies, 95))
        report = dict(backend.metadata, sample_id=sample['id'], audio_variant=branch,
            audio_seconds=seconds, warmup_runs=5, measured_runs=20,
            median_latency_seconds=median, p95_latency_seconds=p95,
            median_rtf=median/seconds, p95_rtf=p95/seconds,
            latencies_seconds=latencies, text=runs[0][0],
            transcript_consistent=len({text for text, elapsed in runs})==1,
            torch_peak_allocated_mib=torch.cuda.max_memory_allocated()/2**20,
            torch_peak_reserved_mib=torch.cuda.max_memory_reserved()/2**20,
            df3=enhancement if branch=='df3' else None,
            notes='Warm synchronized waveform transfer + Fbank + encoder + decode + text. '
                  'Excludes file I/O/startup/model load. Torch memory includes allocator tensors, '
                  'not all CUDA context/driver allocations. Memory peak shared across both variants.')
        save_json(Path(output)/f'gipformer15_{branch}_benchmark.json', report)
        with (Path(output)/f'gipformer15_{branch}_benchmark.csv').open('w',newline='') as f:
            writer = csv.DictWriter(f,fieldnames=['run','latency_seconds','rtf'])
            writer.writeheader()
            writer.writerows(dict(run=i+1,latency_seconds=t,rtf=t/seconds) for i,t in enumerate(latencies))
        print(f'{branch}: median={median:.4f}s, P95={p95:.4f}s, RTF={median/seconds:.5f}',flush=True)


def quality(backend, root, output, manifest, enhancement):
    rows = []
    warmup(backend, load_pair(root, manifest['samples'][0]))
    for index, sample in enumerate(manifest['samples']):
        pair = load_pair(root, sample)
        for branch in (('raw','df3') if index % 2 == 0 else ('df3','raw')):
            text, seconds = backend.recognize(pair[branch])
            row = dict(sample, branch=branch, transcript=text, asr_seconds=seconds,
                       **error_counts(sample['text'], text))
            if branch=='df3':
                row.update(enhancement['samples'][sample['id']])
            rows.append(row)
        print(f'Gipformer {index+1}/{len(manifest["samples"])}: {sample["id"]}',flush=True)
    write_quality_reports(output, rows, dict(backend.metadata,
        manifest_sha256=hashlib.sha256((Path(root)/'manifest.json').read_bytes()).hexdigest(),
        df3_gpu=enhancement['gpu']))


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--root', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--repo', required=True)
    p.add_argument('--icefall-dir', required=True)
    p.add_argument('--model-dir', required=True)
    p.add_argument('--sample-id', default='vi_000_keyboard_events_snr05')
    p.add_argument('--decoding-method', choices=['modified_beam_search','greedy_search'], default='modified_beam_search')
    p.add_argument('--beam-size', type=int, default=4)
    args=p.parse_args()
    if args.beam_size < 1:
        p.error('beam-size must be positive')
    root, output=Path(args.root),Path(args.output)
    if root.resolve()==output.resolve():
        p.error('output must be a different directory from the shared dataset')
    output.mkdir(parents=True,exist_ok=True)
    manifest, enhancement=read_json(root/'manifest.json'),read_json(root/'enhancement.json')
    sha=hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest()
    ids={sample['id'] for sample in manifest['samples']}
    if enhancement.get('manifest_sha256')!=sha or set(enhancement['samples'])!=ids:
        raise RuntimeError('DF3 outputs do not match the dataset; rerun enhancement')
    sample=next((sample for sample in manifest['samples'] if sample['id']==args.sample_id),None)
    if sample is None:
        raise ValueError('Unknown performance sample ID')
    backend=Gipformer(args.repo,args.icefall_dir,args.model_dir,args.decoding_method,args.beam_size)
    performance(backend,root,output,sample,enhancement['samples'][sample['id']])
    quality(backend,root,output,manifest,enhancement)


if __name__=='__main__':
    main()
'''


In [ ]:
quality_worker = WORK_DIR/'quality_benchmark.py'
gip_worker = WORK_DIR/'gipformer_benchmark.py'
quality_worker.write_text(QUALITY_WORKER_SOURCE, encoding='utf-8')
gip_worker.write_text(GIP_WORKER_SOURCE, encoding='utf-8')

def run_logged(command, label, environment=None):
    log_path = OUTPUT_ROOT/f'{label}_stderr.log'
    with log_path.open('w') as log:
        result = subprocess.run([str(x) for x in command], env=environment, stderr=log)
    if result.returncode:
        print(log_path.read_text(errors='replace')[-20000:])
        raise RuntimeError(f'{label} lỗi {result.returncode}; xem {log_path}')
    print('Completed:', label, '| stderr:', log_path)

## 6. Tự tạo hoặc dùng lại dataset
Không upload audio. Nếu dataset đã có, kiểm tra cấu hình và file để dùng lại đúng các bản trộn của Zipformer.
Nếu cấu hình khác, chọn `DATA_ROOT` mới rồi chạy lại từ mục 1 để tránh trộn lẫn hai bộ dữ liệu.
Nguồn: [FLEURS](https://huggingface.co/datasets/google/fleurs), CC BY 4.0;
[DCASE 2016](https://archive.org/details/dcase2016_task2_train_dev), CC BY 3.0 theo metadata archive.

In [ ]:
manifest_path = DATA_ROOT/'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert (manifest.get('schema_version') == 2 and manifest['seed'] == DATA_SEED
            and manifest['speech_clips'] == N_SPEECH), 'Dataset khác cấu hình; chọn DATA_ROOT mới'
    assert manifest['noise_conditions'] == [['keyboard','continuous']] + [
        [name,'events'] for name in ('keyboard','knock','doorslam','keys','pageturn','phone')]
    assert all((DATA_ROOT/sample[key]).exists() for sample in manifest['samples']
               for key in ('input_48k','raw_16k')), 'Dataset thiếu file; khôi phục dữ liệu hoặc chọn DATA_ROOT mới'
    print('Reuse dataset:', manifest_path)
else:
    run_logged([sys.executable, quality_worker, 'prepare', '--root', DATA_ROOT,
                '--count', N_SPEECH, '--seed', DATA_SEED], 'prepare')
    manifest = json.loads(manifest_path.read_text())
from collections import Counter
print('Inputs:', Counter(sample['kind'] for sample in manifest['samples']))
assert any(sample['id']==PERF_SAMPLE_ID for sample in manifest['samples']), 'PERF_SAMPLE_ID không tồn tại'

## 7. Khử nhiễu hoặc dùng lại audio DF3 hợp lệ
Nếu có enhancement cùng checksum manifest, CUDA metadata và đủ file output, dùng lại để hai model nhận đúng cùng audio.
Nếu chưa có, chạy cả dataset trong một worker DF3 CUDA (hai warm-up, một lượt/clip).

In [ ]:
enhancement_path = DATA_ROOT/'enhancement.json'
manifest_sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
enhancement = json.loads(enhancement_path.read_text()) if enhancement_path.exists() else {}
reuse_df3 = (enhancement.get('device') == 'cuda'
    and enhancement.get('manifest_sha256') == manifest_sha
    and set(enhancement.get('samples', {})) == {sample['id'] for sample in manifest['samples']}
    and all((DATA_ROOT/sample['df3_16k']).exists() for sample in manifest['samples']))
if reuse_df3:
    print('Reuse DF3 outputs:', enhancement_path)
else:
    run_logged([DF_PY, quality_worker, 'df3', '--root', DATA_ROOT], 'df3')

## 8. Gipformer CUDA: latency và quality trong cùng một worker
Model load một lần. Kiểm tra CUDA ở parameters, Fbank features và encoder output, synchronize trước/sau mỗi lượt.
Latency: 5 warm-up/nhánh + 20 lượt đo/nhánh, đảo thứ tự raw/DF3; không tính đọc file hay startup/model load.
Quality: cùng nguyên tắc WER/CER và noise-only như Zipformer, một lần nhận dạng/clip; không lấy latency CLI gồm I/O.
Memory dùng Torch allocated/reserved, không phải tổng VRAM gồm context/driver.

In [ ]:
run_logged([GIP_PY, gip_worker, '--root', DATA_ROOT, '--output', OUTPUT_ROOT,
            '--repo', GIP_REPO, '--icefall-dir', ICEFALL_DIR, '--model-dir', model_dir,
            '--sample-id', PERF_SAMPLE_ID, '--decoding-method', DECODING_METHOD,
            '--beam-size', BEAM_SIZE], 'gipformer', environment=GIP_PROCESS_ENV)
for variant in ('raw','df3'):
    report = json.loads((OUTPUT_ROOT/f'gipformer15_{variant}_benchmark.json').read_text())
    print(f"{variant}: median={report['median_latency_seconds']*1000:.2f}ms | "
          f"P95={report['p95_latency_seconds']*1000:.2f}ms | RTF={report['median_rtf']:.5f}")
    print('CUDA checks:', report['cuda_checks'])

## 9. WER/CER theo loại nhiễu và kiểu trộn
Giữ dấu tiếng Việt; NFC/casefold, bỏ dấu câu, không tự đọc số thành chữ.
WER dùng token theo khoảng trắng, CER bỏ khoảng trắng; cộng edit count toàn nhóm trước khi chia.
Kết quả không được so trực tiếp với WER công bố của tác giả vì khác tập con và cách chuẩn hóa.

In [ ]:
quality_results = json.loads((OUTPUT_ROOT/'quality_results.json').read_text())
from IPython.display import display, Markdown
summary = quality_results['summary']
lines = ['| Loại | Nhiễu | Chế độ | SNR | WER raw | WER DF3 | CER raw | CER DF3 | ΔWER (điểm %) |',
         '|---|---|---|---:|---:|---:|---:|---:|---:|']
keys = list(dict.fromkeys((r['kind'], r['noise_type'], r['noise_style'], r['snr_db'])
                         for r in summary if r['kind'] != 'noise_only'))
for kind, noise_type, style, snr in keys:
    pair = {r['branch']: r for r in summary if
            (r['kind'], r['noise_type'], r['noise_style'], r['snr_db']) == (kind, noise_type, style, snr)}
    raw, cleaned = pair['raw'], pair['df3']
    lines.append(f"| {kind} | {noise_type or '—'} | {style} | {'—' if snr is None else snr} | "
                 f"{raw['wer']:.2%} | {cleaned['wer']:.2%} | {raw['cer']:.2%} | {cleaned['cer']:.2%} | "
                 f"{(cleaned['wer']-raw['wer'])*100:+.2f} |")
display(Markdown('\n'.join(lines)))
print('ΔWER âm = giảm lỗi; ΔWER dương = tăng lỗi.')
for r in summary:
    if r['kind'] == 'noise_only':
        print(f"Noise-only / {r['noise_type']} / {r['noise_style']} / {r['branch']}: "
              f"{r['nonempty_transcript_rate']:.0%} sinh chữ ({r['clips']} clip)")

## 10. Chọn bất kỳ mẫu để nghe raw/DF3 và đọc transcript Gipformer
Bộ chọn hiển thị tất cả đầu vào. Reference là transcript chuẩn; raw/df3 Transcript đều do Gipformer sinh.
Audio dùng chung gain playback. Timestamp từng sự kiện nằm trong manifest.

In [ ]:
QUALITY_ROOT = DATA_ROOT
from IPython.display import Audio, display
import soundfile as sf
import numpy as np
import ipywidgets as widgets
from collections import Counter
available = [row for row in quality_results['samples'] if row['branch'] == 'raw']
print('Số đầu vào thực tế:', len(available))
for condition, count in Counter((r['kind'], r['noise_type'], r['noise_style']) for r in available).items():
    print(condition, ':', count)
selector = widgets.Dropdown(
    options=[(row['id'], row['id']) for row in available],
    value=PERF_SAMPLE_ID, description='Mẫu audio:', layout=widgets.Layout(width='95%'))

def play_sample(sample_id):
    global LISTEN_ID
    LISTEN_ID = sample_id
    pair = {row['branch']: row for row in quality_results['samples'] if row['id'] == LISTEN_ID}
    assert len(pair) == 2, 'Chọn LISTEN_ID từ quality_per_clip.csv'
    print('Reference:', pair['raw']['text'] or '(chỉ có nhiễu)')
    for event in pair['raw']['events']:
        print(f"Event {event['start_sample']/48000:.2f}–{event['end_sample']/48000:.2f}s | "
              f"{event['source_file']} | onset={event['onset_region']} | "
              f"speech overlap={event['speech_overlap_fraction']:.0%}")
    playback = {branch: sf.read(QUALITY_ROOT/pair[branch][branch+'_16k'], dtype='float32')
                for branch in ('raw', 'df3')}
    shared_gain = 1/max(1., *(float(np.max(np.abs(audio))) for audio, sr in playback.values()))
    for branch in ('raw', 'df3'):
        row = pair[branch]
        print(branch, '| Transcript:', row['transcript'] or '(rỗng)', '| WER:', row['wer'])
        audio, sr = playback[branch]
        assert sr == 16000
        display(Audio(audio*shared_gain, rate=sr, normalize=False))

player = widgets.interactive_output(play_sample, {'sample_id': selector})
display(selector, player)

## 11. Tải kết quả Gipformer
Mặc định đóng gói toàn bộ audio raw/enhanced (922 WAV nếu đủ 461 đầu vào), báo cáo, metadata, worker và logs.
`INCLUDE_ALL_AUDIO=False` chỉ đóng gói cặp đang nghe. Checkpoint model và các môi trường ảo không nằm trong ZIP.

In [ ]:
import zipfile
from google.colab import files
INCLUDE_ALL_AUDIO = True
bundle = Path('/content/gipformer15_df3_benchmark_results.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_ROOT.iterdir():
        if path.is_file() and path.suffix in ('.csv','.json','.log'):
            z.write(path, arcname=path.name)
    for name in ('manifest.json','enhancement.json'):
        z.write(DATA_ROOT/name, arcname=name)
    z.write(quality_worker, arcname='quality_benchmark.py')
    z.write(gip_worker, arcname='gipformer_benchmark.py')
    for sample in manifest['samples']:
        if not INCLUDE_ALL_AUDIO and sample['id'] != LISTEN_ID:
            continue
        for key in ('raw_16k','df3_16k'):
            z.write(DATA_ROOT/sample[key], arcname=sample[key])
with zipfile.ZipFile(bundle) as check:
    audio_count = sum(name.endswith('.wav') for name in check.namelist())
assert audio_count == (2*len(manifest['samples']) if INCLUDE_ALL_AUDIO else 2)
print('WAV files:', audio_count, '| ZIP:', bundle, f'({bundle.stat().st_size/2**20:.1f} MiB)')
files.download(str(bundle))